<p align="center">
<a href="https://duckietown.com"><img src="../assets/images/dtlogo.png" alt="Duckietown Logo" width="30%"></a>
</p>


<style>
.lx-table { margin:1.5em auto; text-align:left; }
.lx-table caption { caption-side:top; font-size:0.9em; margin-bottom:0.6em; text-align:center; }
.lx-figure { margin:1.5em auto; text-align:center; }
.lx-figure figcaption { font-size:0.9em; margin-top:0.6em; text-align:center; }
table { margin:0 auto 1.5em; }
p:has(> a[id^='table-']) { margin:0; }
p:has(> a[id^='table-']) + p, a[id^='table-'] + p { font-size:0.9em; margin:1.5em 0 0.6em; text-align:center; }
p.lx-figure { margin:1.5em auto 0; }
p.lx-figure + p { font-size:0.9em; margin:0.6em 0 1.5em; text-align:center; }
</style>

# Time-of-Flight (ToF) Sensor Fundamentals

Measuring distance from objects is a key function for every robot. The Duckiedrone achieves this with five small dedicated laser rangefinders. A downward sensor gives the flight controller a height reference above the surface below, while the other four face upward, forward, left, and right.

[Figure 1](#figure-1) shows the five sensor directions.

<figure id="figure-1" class="lx-figure" style="width:60%;">
  <img src="../assets/_images/tof_sensor_layout.svg" style="width:100%;" />
  <figcaption>
    Figure 1: ToF sensor layout, shown using forward-right-down axes:
    x forward, y right, and z down.
  </figcaption>
</figure>

Each rangefinder is a __direct time-of-flight (dToF) sensor__: it measures (1D) distance from the time light takes to reach a surface and return. All five sensors mounted on the Duckiedrone use the STMicroelectronics [VL53L1X](https://www.st.com/en/imaging-and-photonics-solutions/vl53l1x.html). A useful distance reading depends on the light returned by the scene. A dark surface, a distant target, or strong ambient infrared light can make a measurement less reliable or prevent detection. This notebook begins with one reported distance, then uses a photon-count model and a stationary recording to investigate those limits.

The diagram uses the forward-right-down convention introduced for PX4 in the [Duckietown IMU LX](https://docs.duckietown.com/ente/opmanual-dd24/50-learning-experiences/supported-lxs/sensors-inertial-measurement-unit-imu.html). A ToF range is a single distance, rather than three components expressed along body axes.

This notebook addresses the following questions:

- How does a rangefinder work, and in particular, how does the VL53L1X use light to estimate distance?

- Why do repeated measurements vary when the Duckiedrone remains stationary?

- How do the target, ambient light, and measurement settings affect usable range?

- Why does a direct range measurement behave differently from a position estimate obtained by integrating acceleration?

## Notation

[Table 1](#table-1) defines the quantities used in the distance equation.

<a id="table-1"></a>

Table 1: Quantities used to describe a time-of-flight measurement.

| Symbol | Meaning | Units |
| --- | --- | --- |
| $d$ | True distance from the sensor to a single reflecting target in the idealized measurement | $\mathrm{m}$ |
| $\tilde{d}$ | Reported measurement of that distance | $\mathrm{m}$ |
| $c$ | Speed of light in air, approximately $3 \times 10^8$ | $\mathrm{m/s}$ |
| $\Delta t$ | Time between emission and return of the light | $\mathrm{s}$ |

## What a time-of-flight sensor measures

[Figure 2](#figure-2) illustrates the light's outward and return paths.

<figure id="figure-2" class="lx-figure" style="width:420px;">
  <img src="../assets/_images/Time_of_flight.svg" style="width:100%;" />
  <figcaption>
    Figure 2: Light travels from the sensor to the target and back.
    The graphic labels the round-trip duration t; the equation below uses
    delta t for that same interval.
    <a href="https://en.wikipedia.org/wiki/Time-of-flight_camera#/media/File:20200501_Time_of_flight.svg">Image source</a>.
  </figcaption>
</figure>

For a stationary target and approximately coincident emitting and receiving positions, the light travels a total distance of $2d$. Its round-trip time therefore gives:

<span id="equation-1"></span>

$$
d = \frac{c\,\Delta t}{2}.
\tag{1}
$$

The sensor reports an estimate of this distance, $\tilde{d}$, which can differ from the physical distance because of measurement noise, calibration offsets, and the target and lighting conditions.

The VL53L1X has a typical full field of view of about $27^\circ$. Its programmable region of interest (ROI) selects part of the receiving array and can narrow that field of view. The Duckiedrone driver advertises the full field of view. Several surfaces within it can contribute returned light to the single reported range, so the single-target picture in Figure 2 assumes one dominant reflecting surface.

For a level Duckiedrone above a clear, flat floor, the downward reading approximates the __sensor__'s height above that floor: to determine the height of, for example, the center of mass of the Duckiedrone, it is necessary to account for the mounting offset.

## Gathering ToF measurements from a Duckiedrone

The following message was captured from a physical Duckiedrone resting on the floor:

```shell
header:
  stamp:
    sec: 1787340653
    nanosec: 105715314
  frame_id: DUCKIEDRONE_NAME/tof/bottom
radiation_type: 1
field_of_view: 0.471238911151886
min_range: 0.03999999910593033
max_range: 3.5999999046325684
range: 0.017999999225139618
variance: 0.0
```

The message has type `sensor_msgs/msg/Range`, and its `range` field reports approximately $0.018\,\mathrm{m}$, or $18\,\mathrm{mm}$. The accompanying bounds are approximately $0.04\,\mathrm{m}$ and $3.6\,\mathrm{m}$, as advertised by the driver's long-range profile. Consumers should discard readings outside these bounds, including this example. Accuracy within the interval depends on the target, lighting, and measurement settings. [Duckiedrone ToF Sensor Architecture](./3-duckiedrone-tof-architecture.ipynb) explains the message fields and their source.

The `variance` value of zero means that the variance is unknown, rather than that repeated measurements are identical.

To inspect a live reading, connect to a physical or virtual Duckiedrone and open a terminal inside the `ros2-tof-bottom` Docker container.

### Connecting to the Duckiedrone

Replace `DUCKIEDRONE_NAME` in the commands below with the configured Duckiedrone name. For a physical Duckiedrone, SSH into it with:

```bash
ssh duckie@DUCKIEDRONE_NAME.local
```

For a virtual Duckiedrone, start it with `dts duckiebot virtual start DUCKIEDRONE_NAME` if it is not running, then connect:

```bash
dts duckiebot virtual connect DUCKIEDRONE_NAME
```

### Find the containers that publish the ToF

```bash
docker ps --format '{{.Names}}'
```

The standard downward-ToF pair is `driver-tof-bottom` and `ros2-tof-bottom`. The former container includes the drivers for reading the sensor, while the latter bridge publishes its ROS 2 topic.

### Open a shell inside the bridge

To open a terminal inside the container with the measurements:

```bash
docker exec -it ros2-tof-bottom bash
```

On a virtual Duckiedrone, simulated readings arrive only while the Duckiematrix is running and [the Duckiedrone is attached to an entity](https://docs.duckietown.com/ente/opmanual-dd24/50-learning-experiences/supported-lxs/sensors-time-of-flight.html#attaching-the-virtual-duckiedrone-to-the-duckiematrix). Its ranges are computed from the simulated scene; the current simulated sensor adds no measurement noise.

Find the topic and its exact name with:

```bash
ros2 topic list
```

and show one message with:

```bash
ros2 topic echo --once /DUCKIEDRONE_NAME/bottom_tof_driver_node/range
```

The command prints one message and exits after receiving it. The timestamp, frame identifier, and range can differ from the example.

## How the distance is measured

The round-trip times involved are very short. From [Equation (1)](#equation-1), a distance change of $1\,\mathrm{mm}$ corresponds to:

$$
\Delta t =
\frac{2 \times 10^{-3}}{3 \times 10^8}
\approx 6.7 \times 10^{-12}\,\mathrm{s},
$$

or about $6.7$ picoseconds. Such precise timing measurements are possible using [time-to-digital conversion (TDC)](https://en.wikipedia.org/wiki/Time-to-digital_converter) circuitry. The VL53L1X combines information from many photon detections to estimate a distance measurement. Its receiver contains an array of __single-photon avalanche diodes ([SPAD](https://en.wikipedia.org/wiki/Single-photon_avalanche_diode)s)__, where detected photons can trigger electrical pulses. The ToF sensor repeatedly emits infrared laser pulses, some of whose photons return after reflection from surfaces in the scene. Ambient light contributes additional detections.

The chip's internal processing combines these detections into a range estimate. ST classifies the VL53L1X as a __sigma-delta ranging__ sensor in [application note AN5231](https://www.st.com/resource/en/application_note/an5231-cover-glass-guidelines-for-the-singlezone-timeofflight-sensor-stmicroelectronics.pdf), Section 4.1.1. The Raspberry Pi reads the resulting distance and status over I2C; the driver and bridge then make the reading available through ROS 2.

The distance measurement is finally reported in millimeter increments.

## A model of measurement generation

The following activity makes the effects of limited photon counts visible. It uses a generic histogram-based teaching model with an explicit peak-selection rule.

The model groups detected photons by arrival time. For convenient plotting, each time bin is converted to a distance using Equation (1). Light returning from one target produces a broad peak, while ambient light contributes a background across the bins.

The model assumes:

- One stationary target fills the sensor's field of view.

- The expected return decreases with the square of distance and increases with target reflectivity.

- Background detections are distributed uniformly across the arrival-time bins.

- Longer collection time increases both signal and background counts.

The constants are chosen for illustration, so the detection distances and errors describe this teaching model. Predicting a physical VL53L1X's performance requires characterization under the intended measurement conditions.

Run the next two cells to display one modeled measurement.


In [ ]:
# The sensor model. This cell defines functions and produces no plot.
import numpy as np

BIN_WIDTH_M = 0.01
BIN_DISTANCE = (np.arange(400) + 0.5) * BIN_WIDTH_M
PULSE_SPREAD_M = 0.045
REFERENCE_TIME_MS = 33.0
SIGNAL_GAIN = 13000.0
DETECTION_SCORE = 6.0


def photon_histogram(
    distance_m=2.5,
    reflectivity=0.5,
    background_per_bin=2.0,
    collection_time_ms=33,
    seed=0,
):
    """Generate a generic photon-count model, not VL53L1X output."""
    scale = collection_time_ms / REFERENCE_TIME_MS
    background = background_per_bin * scale
    signal = SIGNAL_GAIN * reflectivity * scale / distance_m**2

    bump = np.exp(
        -0.5 * ((BIN_DISTANCE - distance_m) / PULSE_SPREAD_M)**2
    )
    expected = background + signal * bump / (
        np.sqrt(2 * np.pi) * PULSE_SPREAD_M / BIN_WIDTH_M
    )

    rng = np.random.default_rng(seed)
    return BIN_DISTANCE, rng.poisson(expected)


def find_peak(bins, counts):
    """Apply the teaching model's chosen detection rule."""
    background = float(np.median(counts))
    peak = int(np.argmax(counts))

    score = (counts[peak] - background) / np.sqrt(
        counts[peak] + background + 1.0
    )
    estimate = float(bins[peak]) if score >= DETECTION_SCORE else None
    return estimate, background, float(score)

In [ ]:
import matplotlib.pyplot as plt


def show_measurement(
    distance_m=2.5,
    reflectivity=0.5,
    background_per_bin=2.0,
    collection_time_ms=33,
    seed=0,
):
    bins, counts = photon_histogram(
        distance_m,
        reflectivity,
        background_per_bin,
        collection_time_ms,
        seed,
    )
    estimate, background, score = find_peak(bins, counts)

    fig, ax = plt.subplots(figsize=(9, 4))
    ax.step(bins, counts, where="mid", label="modeled photon counts")
    ax.axhline(
        background, color="gray", ls=":", label="estimated background"
    )
    ax.axvline(
        distance_m, color="black", ls="--", label="target distance"
    )

    if estimate is None:
        result = "no detection"
    else:
        ax.axvline(estimate, color="tab:red", label="model estimate")
        result = (
            f"estimate {estimate:.3f} m; "
            f"error {estimate - distance_m:+.3f} m"
        )

    ax.set_title(f"Generic photon model: {result}; score {score:.1f}")
    ax.set_xlabel("round-trip arrival time expressed as distance [m]")
    ax.set_ylabel("photons per bin")
    ax.set_xlim(0, 4)
    ax.legend()
    ax.grid(alpha=0.3)
    fig.tight_layout()
    plt.show()


show_measurement()

<style>
.lx-table { margin:1.5em auto; text-align:left; }
.lx-table caption { caption-side:top; font-size:0.9em; margin-bottom:0.6em; text-align:center; }
.lx-figure { margin:1.5em auto; text-align:center; }
.lx-figure figcaption { font-size:0.9em; margin-top:0.6em; text-align:center; }
table { margin:0 auto 1.5em; }
p:has(> a[id^='table-']) { margin:0; }
p:has(> a[id^='table-']) + p, a[id^='table-'] + p { font-size:0.9em; margin:1.5em 0 0.6em; text-align:center; }
p.lx-figure { margin:1.5em auto 0; }
p.lx-figure + p { font-size:0.9em; margin:0.6em 0 1.5em; text-align:center; }
</style>

The blue trace shows modeled photon counts from the uniform background and a concentrated target return. The black dashed line marks the prescribed target distance, while the red line, when a detection is reported, marks the model estimate obtained by selecting the tallest bin. The distance between the red and black lines is the model's measurement error.

The horizontal dotted line estimates the background using the median bin count across all distances. A target must produce a peak sufficiently far above the background fluctuations before the model reports a detection.

[Table 2](#table-2) defines the quantities used by this detection rule.

<a id="table-2"></a>

Table 2: Quantities used by the teaching model's detection rule.

| Symbol | Meaning |
| --- | --- |
| $N$ | Photon count in the tallest bin |
| $B$ | Estimated background count per bin |
| $S$ | Dimensionless detection score |

For independent photon arrivals, a Poisson count has variance equal to its expected count. This motivates comparing the excess above background with a scale proportional to the square root of the counts. The model uses:

<span id="equation-2"></span>

$$
S = \frac{N-B}{\sqrt{N+B+1}}.
\tag{2}
$$

The added $1$ keeps the denominator nonzero. The model reports a detection when $S \geq 6$.

This illustrative score measures peak prominence under the chosen rule.

## The chip's settings

The model's collection time illustrates why allowing more time for a measurement can help. The actual [Duckiedrone (bottom ToF sensor) driver](https://github.com/duckietown/dt-duckiebot-interface/blob/ente/packages/tof_driver/config/duckiedrone/bottom/default.yaml) exposes distance-mode and timing settings that affect the VL53L1X's ranging behavior.

The relevant defaults and an optional period setting are:

```yaml
sensor_name: "bottom"
sensor_model: "VL53L1X"

# Requested publication frequency, Hz.
frequency: 30

# Distance mode: SHORT_RANGE or LONG_RANGE.
mode: "LONG_RANGE"

# Time allocated to one ranging operation, ms.
timing_budget: 33

# Interval between measurement starts, ms; at least timing_budget.
# Derived from frequency and timing_budget when omitted or null.
# inter_measurement_period: 33
```

### Distance `mode`

`SHORT_RANGE` supports distances up to approximately 1.3 m and favors operation under stronger ambient light, while `LONG_RANGE` supports distances up to approximately 4 m under favorable conditions. These modes change the chip's internal settings.

### Timing budget

The __timing budget__ is the time allocated to one ranging operation:

1. Emit IR light

2. Record reflections

A longer budget can improve repeatability and usable range because the sensor has more time to collect information. The Duckietown driver accepts budgets of 15, 20, 33, 50, 100, 200, and 500 ms, with 15 ms available only in short mode. The downward sensor is configured with a 33 ms budget.

The __inter-measurement period__ determines the interval between measurement starts. The driver requires it to be at least as long as the timing budget. A 200 ms budget would, therefore, permit at most five measurement starts per second, before considering other timing constraints.

[Duckiedrone ToF Sensor Architecture](./3-duckiedrone-tof-architecture.ipynb#changing-any-of-this) connects these settings to the driver configuration and observed message rate.


### Predict, then check

The widget below changes the generic model's target distance, reflectivity, background level, and collection time. The background control specifies the expected count per bin at the reference collection time of 33 ms.

The `random draw` control selects another realization of the photon counts. Keeping all controls unchanged reproduces the same result.

Start with a distance of 2.5 m, reflectivity 0.5, background 2 counts per bin, collection time 33 ms, and random draw 0.


In [ ]:
from ipywidgets import Dropdown, FloatSlider, IntSlider, interact

LABEL_STYLE = {"description_width": "170px"}

interact(
    show_measurement,
    distance_m=FloatSlider(
        value=2.5,
        min=0.5,
        max=3.5,
        step=0.1,
        description="distance [m]",
        style=LABEL_STYLE,
        continuous_update=False,
    ),
    reflectivity=FloatSlider(
        value=0.5,
        min=0.05,
        max=1.0,
        step=0.05,
        description="reflectivity",
        style=LABEL_STYLE,
        continuous_update=False,
    ),
    background_per_bin=FloatSlider(
        value=2.0,
        min=0.0,
        max=150.0,
        step=1.0,
        description="background [counts/bin]",
        style=LABEL_STYLE,
        continuous_update=False,
    ),
    collection_time_ms=Dropdown(
        options=[20, 33, 50, 100, 200],
        value=33,
        description="collection time [ms]",
        style=LABEL_STYLE,
    ),
    seed=IntSlider(
        value=0,
        min=0,
        max=20,
        step=1,
        description="random draw",
        style=LABEL_STYLE,
        continuous_update=False,
    ),
);

<style>
.lx-table { margin:1.5em auto; text-align:left; }
.lx-table caption { caption-side:top; font-size:0.9em; margin-bottom:0.6em; text-align:center; }
.lx-figure { margin:1.5em auto; text-align:center; }
.lx-figure figcaption { font-size:0.9em; margin-top:0.6em; text-align:center; }
table { margin:0 auto 1.5em; }
p:has(> a[id^='table-']) { margin:0; }
p:has(> a[id^='table-']) + p, a[id^='table-'] + p { font-size:0.9em; margin:1.5em 0 0.6em; text-align:center; }
p.lx-figure { margin:1.5em auto 0; }
p.lx-figure + p { font-size:0.9em; margin:0.6em 0 1.5em; text-align:center; }
</style>

Predict each change's effect on the model before moving the control sliders.

1. __Distance.__ Move the target from 2.5 m to 1.0 m. The peak moves toward the origin and becomes taller. Under the model's inverse-square assumption, the expected signal increases by $(2.5/1.0)^2 = 6.25$. Return to 2.5 m.

2. __Reflectivity.__ Reduce reflectivity from 0.5 to 0.2. The expected signal becomes 40% of its previous value, while the background remains unchanged. Try several random draws and observe whether the model reports a detection. Return reflectivity to 0.5 and random draw to 0.

3. __Background.__ Increase the background from 2 to 150 counts per bin. The expected target contribution remains unchanged, but the background and its fluctuations increase. Compare the detection score with the initial result and try several random draws.

4. __Collection time.__ Keep the higher background and increase collection time from 33 to 200 ms. Both signal and background counts increase by approximately $200/33 \approx 6.1$. Their random fluctuations grow more slowly than their counts, making the target easier to distinguish in this model.

5. __Repeated measurements.__ Compare the spread of estimates across several random draws at each collection time to investigate repeatability. Restore the initial settings when finished.

These changes illustrate how the amount of useful returned light affects a measurement.

## The same measurement on a physical Duckiedrone

The model isolates selected effects under controlled assumptions. The supplied recording shows how a physical sensor's output varied during one stationary experiment.

A physical Duckiedrone rested on the floor with its motors off while its downward ToF readings were recorded. The file [assets/data/tof_still_3min.csv](../assets/data/tof_still_3min.csv) contains 5,551 samples spanning approximately 185 s. Its columns are `t`, in seconds from the start, and `range`, in meters.

The reported ranges are below the advertised 4 cm minimum. This recording shows their variation during one stationary experiment; assessing accuracy also requires an independently measured sensor-to-floor distance.

Run the next cell to inspect the samples and their distribution.


In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt  # noqa: F811
import numpy as np

data_path = Path("assets/data/tof_still_3min.csv")
if not data_path.exists():
    data_path = Path("../assets/data/tof_still_3min.csv")

recording = np.genfromtxt(data_path, delimiter=",", names=True)
t_s = recording["t"]
range_mm = recording["range"] * 1000.0

duration_s = t_s[-1] - t_s[0]
rate_hz = (len(t_s) - 1) / duration_s
mean_mm = float(range_mm.mean())

# Describe the supplied samples using the population standard deviation.
std_mm = float(range_mm.std(ddof=0))

first_minute = range_mm[t_s < t_s[0] + 60.0]
last_minute = range_mm[t_s > t_s[-1] - 60.0]
mean_change_mm = float(last_minute.mean() - first_minute.mean())

consecutive_corr = float(
    np.corrcoef(range_mm[:-1], range_mm[1:])[0, 1]
)

fig, (ax_t, ax_h) = plt.subplots(
    1, 2, figsize=(12, 4), width_ratios=[2, 1]
)

ax_t.plot(
    t_s,
    range_mm,
    linestyle="none",
    marker=".",
    markersize=2,
    color="tab:blue",
)
ax_t.axhline(
    mean_mm, color="black", ls="--", label=f"mean {mean_mm:.2f} mm"
)
ax_t.set_xlabel("time [s]")
ax_t.set_ylabel("reported range [mm]")
ax_t.set_title(
    f"{len(range_mm)} samples over {duration_s:.1f} s"
)
ax_t.legend()
ax_t.grid(alpha=0.3)

edges = np.arange(
    range_mm.min() - 0.5, range_mm.max() + 1.5, 1.0
)
ax_h.hist(
    range_mm, bins=edges, color="tab:blue", edgecolor="white"
)
ax_h.set_xlabel("reported range [mm]")
ax_h.set_ylabel("number of samples")
ax_h.set_title(f"standard deviation {std_mm:.2f} mm")
ax_h.grid(alpha=0.3)

fig.tight_layout()
plt.show()

print(f"Recorded sample rate: {rate_hz:.2f} Hz")
print(
    f"Reported values: {range_mm.min():.0f} to "
    f"{range_mm.max():.0f} mm"
)
print(f"Mean: {mean_mm:.2f} mm")
print(f"Standard deviation (ddof=0): {std_mm:.2f} mm")
print(f"First-minute mean: {first_minute.mean():.2f} mm")
print(f"Last-minute mean: {last_minute.mean():.2f} mm")
print(f"Last minus first minute: {mean_change_mm:+.2f} mm")
print(f"Consecutive-sample correlation: {consecutive_corr:.3f}")


<style>
.lx-table { margin:1.5em auto; text-align:left; }
.lx-table caption { caption-side:top; font-size:0.9em; margin-bottom:0.6em; text-align:center; }
.lx-figure { margin:1.5em auto; text-align:center; }
.lx-figure figcaption { font-size:0.9em; margin-top:0.6em; text-align:center; }
table { margin:0 auto 1.5em; }
p:has(> a[id^='table-']) { margin:0; }
p:has(> a[id^='table-']) + p, a[id^='table-'] + p { font-size:0.9em; margin:1.5em 0 0.6em; text-align:center; }
p.lx-figure { margin:1.5em auto 0; }
p.lx-figure + p { font-size:0.9em; margin:0.6em 0 1.5em; text-align:center; }
</style>

### ToF sensor precision and accuracy

The left plot displays each recorded sample as a point. The right plot groups the reported distances into a histogram, with its bars counting range measurements.

The readings span 11 to 25 mm, with a mean of approximately 18.52 mm and a standard deviation of 2.07 mm. The reported values change in steps of 1 mm, the sensor's reporting resolution.

This distribution characterizes measurement precision, or repeatability, under the recorded conditions: long-range mode, a close target, and the surface and ambient lighting used in the experiment.

Accuracy describes agreement with the true distance. [Measuring ToF sensor accuracy](./2-measuring-tof-sensor-accuracy.ipynb) investigates it by comparing readings with an independently measured reference.

### The meaning of measurements

The [IMU LX](https://github.com/duckietown/lx-dd-sensors-imu) used acceleration measurements to construct velocity and position estimates of the Duckiedrone by integration, that is, using the dead-reckoning estimation algorithm. This approach is straightforward to implement, but measurement errors accumulate through integration and can cause the estimates to drift.

Each ToF reading estimates distance from a new ranging operation. Its error depends on that measurement and the sensing conditions, while an integrated position estimate also carries accumulated errors from earlier steps.


## What "maximum range" means

A target becomes harder to detect as its useful return, that is, the light bouncing off the target that is intercepted by the sensing array, weakens relative to background and measurement uncertainty. Target reflectivity, distance, ambient infrared light, and measurement settings therefore influence the usable range.

ST's published range figures apply to stated test conditions. A target that fills the field of view in a dark room can give different results from a smaller or less reflective target in strong ambient light.

The range accepted by the Duckietown software is another limit. In `LONG_RANGE`, the current driver uses 3.6 m as its upper bound and discards larger results. The ROS 2 bridge represents an unavailable or rejected range with `999.0`. This value is outside the message's valid range and is not a measured distance.


## What a ToF sensor does not measure

The range must be interpreted together with the scene and sensor orientation.

- __Altitude above a fixed reference.__ Flying over a table changes the downward range even if the Duckiedrone remains at the same altitude.

- __The identity of the reflecting surface.__ Several surfaces can contribute within the field of view, while the sensor reports one range.

- __Vertical clearance in every orientation.__ Tilting the Duckiedrone changes the measurement geometry. In an ideal central-ray picture above a flat floor, the ray follows a longer, slanted path; the physical reading also depends on which surfaces contribute light.

- __A distance whenever a target is present.__ A weak or rejected return can leave the software without a usable range.

## Summary and next steps

A ToF sensor obtains a direct distance observation from returned light. The measurement can vary even in a stationary scene, and a returned number must be checked against the interface's validity limits.

The photon-count model illustrated how signal, background, and collection time affect a generic ranging estimate. The physical recording characterized the spread of reported ranges; assessing their accuracy requires comparison with an independently measured distance.

[Measuring ToF sensor accuracy](./2-measuring-tof-sensor-accuracy.ipynb) makes that comparison with a stationary target, a ruler measurement, and a recording from the physical Duckiedrone.

## Further reading

- The [VL53L1X datasheet](https://www.st.com/resource/en/datasheet/vl53l1x.pdf) specifies the sensor's field of view and ranging performance under stated conditions.

- ST's [VL53L1X ultra lite driver manual](https://www.st.com/resource/en/user_manual/um2510-a-guide-to-using-the-vl53l1x-ultra-lite-driver-stmicroelectronics.pdf) explains distance modes, timing budgets, and inter-measurement periods.

- The [sensor_msgs/Range definition](https://github.com/ros2/common_interfaces/blob/jazzy/sensor_msgs/msg/Range.msg) specifies the message fields and treatment of readings outside its limits.

## Checkpoint

Run the self-check in the next cell. Write or select a response before revealing the answer.


In [ ]:
import sys
from pathlib import Path

working_directory = Path.cwd()
parent_directory = working_directory.parent
if (parent_directory / "packages").is_dir():
    parent_directory_path = str(parent_directory)
    sys.path.insert(0, parent_directory_path)

from packages.checkpoint_self_check import display_checkpoint_self_checks

display_checkpoint_self_checks()